# 18 - Build Independent Dataset (1 Study Per Patient) — Normalised Images

Same as NB 18 but using the normalised radiomics features from NB 12/13 normalised.
Selects exactly one study per patient to create a truly independent dataset.

In [1]:
import os
import pandas as pd

print("Imports OK")

Imports OK


In [2]:
df = pd.read_csv(os.path.join("reports", "13_merged_radiomics_clinical_normalised.csv"))

print(f"Full dataset (normalised): {len(df)} studies, {df['patient_id'].nunique()} patients")
print(f"Class balance: {(df['rejection']==0).sum()} no-rej, {(df['rejection']==1).sum()} rej")
print(f"Studies per patient: min={df.groupby('patient_id').size().min()}, "
      f"max={df.groupby('patient_id').size().max()}, "
      f"mean={df.groupby('patient_id').size().mean():.1f}")

Full dataset (normalised): 137 studies, 55 patients
Class balance: 98 no-rej, 39 rej
Studies per patient: min=1, max=6, mean=2.5


## Patients with multiple outcomes

14 patients have both rejection and no-rejection studies. For these, we pick the first rejection study. For the other 41 patients (single outcome), we pick the first study.

In [3]:
outcomes_per_patient = df.groupby("patient_id")["rejection"].nunique()
patients_both = outcomes_per_patient[outcomes_per_patient == 2].index
patients_single = outcomes_per_patient[outcomes_per_patient == 1].index

print(f"Patients with single outcome: {len(patients_single)}")
print(f"Patients with both outcomes:  {len(patients_both)}")
print(f"Total patients: {len(patients_single) + len(patients_both)}")
print()
print("Patients with both outcomes:")
for pid in sorted(patients_both):
    sub = df[df["patient_id"] == pid][["study_id", "rejection", "motivo"]]
    sub = sub.sort_values("study_id")
    studies_str = ", ".join(
        f"{row['study_id']}(rej={int(row['rejection'])})" for _, row in sub.iterrows()
    )
    print(f"  Patient {pid}: {studies_str}")

Patients with single outcome: 41
Patients with both outcomes:  14
Total patients: 55

Patients with both outcomes:
  Patient 1: 01_01(rej=0), 01_02(rej=1), 01_03(rej=1), 01_04(rej=0), 01_05(rej=0)
  Patient 6: 06_01(rej=0), 06_02(rej=1), 06_03(rej=0), 06_04(rej=0)
  Patient 9: 09_01(rej=1), 09_02(rej=1), 09_03(rej=0), 09_04(rej=1), 09_05(rej=0)
  Patient 12: 12_01(rej=1), 12_02(rej=0), 12_03(rej=1), 12_04(rej=1)
  Patient 14: 14_01(rej=0), 14_02(rej=0), 14_03(rej=1), 14_04(rej=0), 14_05(rej=1), 14_06(rej=1)
  Patient 16: 16_01(rej=0), 16_02(rej=1)
  Patient 18: 18_01(rej=1), 18_02(rej=1), 18_03(rej=0)
  Patient 19: 19_01(rej=1), 19_02(rej=0), 19_03(rej=1), 19_04(rej=1)
  Patient 29: 29_01(rej=0), 29_02(rej=1), 29_03(rej=1), 29_04(rej=1)
  Patient 36: 36_01(rej=1), 36_02(rej=0)
  Patient 37: 37_01(rej=0), 37_02(rej=1), 37_03(rej=0)
  Patient 40: 40_01(rej=1), 40_02(rej=0), 40_03(rej=0)
  Patient 41: 41_01(rej=0), 41_02(rej=1), 41_03(rej=0)
  Patient 55: 55_01(rej=1), 55_02(rej=0), 55_03

## Select one study per patient

In [4]:
selected_rows = []

for pid, group in df.groupby("patient_id"):
    group_sorted = group.sort_values("study_id")
    outcomes = group["rejection"].unique()

    if len(outcomes) == 1:
        # Single outcome: pick the first study
        pick = group_sorted.iloc[0]
    else:
        # Both outcomes: pick the first rejection study
        rej_studies = group_sorted[group_sorted["rejection"] == 1]
        pick = rej_studies.iloc[0]

    selected_rows.append(pick)

independent_df = pd.DataFrame(selected_rows).reset_index(drop=True)
independent_df = independent_df.sort_values("study_id").reset_index(drop=True)

print(f"Independent dataset: {len(independent_df)} studies")
print(f"Class balance: {(independent_df['rejection']==0).sum()} no-rej, "
      f"{(independent_df['rejection']==1).sum()} rej")
print(f"Rejection rate: {independent_df['rejection'].mean()*100:.1f}%")

Independent dataset: 55 studies
Class balance: 34 no-rej, 21 rej
Rejection rate: 38.2%


## Validation checks

In [5]:
# Check 1: exactly 1 study per patient
assert independent_df["patient_id"].nunique() == len(independent_df), "Duplicate patients!"
print("Check 1 passed: exactly 1 study per patient")

# Check 2: all patients represented
assert set(independent_df["patient_id"]) == set(df["patient_id"]), "Missing patients!"
print("Check 2 passed: all 55 patients represented")

# Check 3: for patients with both outcomes, we picked a rejection study
for pid in patients_both:
    row = independent_df[independent_df["patient_id"] == pid]
    assert row["rejection"].values[0] == 1, f"Patient {pid}: expected rejection study"
print(f"Check 3 passed: all {len(patients_both)} dual-outcome patients have rejection study selected")

# Check 4: columns match the full dataset
assert list(independent_df.columns) == list(df.columns), "Column mismatch!"
print(f"Check 4 passed: columns match ({len(independent_df.columns)} columns)")

# Check 5: no NaN in radiomics features
feature_cols = [c for c in independent_df.columns if c.startswith("original_")]
n_nan = independent_df[feature_cols].isna().sum().sum()
print(f"Check 5 passed: {n_nan} NaN values in {len(feature_cols)} feature columns")

Check 1 passed: exactly 1 study per patient
Check 2 passed: all 55 patients represented
Check 3 passed: all 14 dual-outcome patients have rejection study selected
Check 4 passed: columns match (100 columns)
Check 5 passed: 0 NaN values in 93 feature columns


In [6]:
# Show the selected studies with selection reason
print(f"{'study_id':>10}  {'patient':>7}  {'rej':>3}  {'motivo':>6}  {'reason'}")
print("-" * 55)
for _, row in independent_df.iterrows():
    pid = int(row["patient_id"])
    if pid in patients_both:
        reason = "first rejection (both outcomes)"
    else:
        reason = "first study (single outcome)"
    print(f"{row['study_id']:>10}  {pid:>7}  {int(row['rejection']):>3}  "
          f"{int(row['motivo']):>6}  {reason}")

  study_id  patient  rej  motivo  reason
-------------------------------------------------------
     01_02        1    1       2  first rejection (both outcomes)
     02_01        2    1       4  first study (single outcome)
     03_01        3    1       4  first study (single outcome)
     04_01        4    0       2  first study (single outcome)
     05_01        5    0       1  first study (single outcome)
     06_02        6    1       2  first rejection (both outcomes)
     07_01        7    0       1  first study (single outcome)
     08_01        8    1       5  first study (single outcome)
     09_01        9    1       4  first rejection (both outcomes)
     10_01       10    0       1  first study (single outcome)
     11_02       11    0       4  first study (single outcome)
     12_01       12    1       1  first rejection (both outcomes)
     13_01       13    0       1  first study (single outcome)
     14_03       14    1       4  first rejection (both outcomes)
     1

## Compare full vs independent dataset

In [7]:
print(f"{'':25s}  {'Full':>8}  {'Independent':>12}")
print("-" * 50)
print(f"{'Studies':25s}  {len(df):>8}  {len(independent_df):>12}")
print(f"{'Patients':25s}  {df['patient_id'].nunique():>8}  {independent_df['patient_id'].nunique():>12}")
print(f"{'No-rejection':25s}  {(df['rejection']==0).sum():>8}  {(independent_df['rejection']==0).sum():>12}")
print(f"{'Rejection':25s}  {(df['rejection']==1).sum():>8}  {(independent_df['rejection']==1).sum():>12}")
print(f"{'Rejection rate':25s}  {df['rejection'].mean()*100:>7.1f}%  {independent_df['rejection'].mean()*100:>11.1f}%")
print()
print("Motivo distribution:")
for m in sorted(df["motivo"].unique()):
    n_full = (df["motivo"] == m).sum()
    n_ind = (independent_df["motivo"] == m).sum()
    print(f"  motivo {m}: full={n_full}, independent={n_ind}")

                               Full   Independent
--------------------------------------------------
Studies                         137            55
Patients                         55            55
No-rejection                     98            34
Rejection                        39            21
Rejection rate                28.5%         38.2%

Motivo distribution:
  motivo 1: full=30, independent=23
  motivo 2: full=37, independent=14
  motivo 3: full=27, independent=0
  motivo 4: full=23, independent=17
  motivo 5: full=20, independent=1


## Save

In [8]:
output_path = os.path.join("reports", "18_independent_dataset_normalised.csv")
independent_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Shape: {independent_df.shape}")
print(f"Columns: {len(independent_df.columns)} (same as 13_merged_radiomics_clinical_normalised.csv)")

Saved to reports/18_independent_dataset_normalised.csv
Shape: (55, 100)
Columns: 100 (same as 13_merged_radiomics_clinical_normalised.csv)


## Summary

Created an independent dataset with exactly 1 study per patient (55 studies from 55 patients).

Selection rule:
- Single-outcome patients (41): first study by study_id
- Dual-outcome patients (14): first rejection study by study_id

This dataset is used for NB 19 (stats) and NB 20 (ML) to validate results without repeated-measures bias.